# The Fannie Mae Loan-Performance Data Bible

**One notebook that documents the entire dataset**: every column's meaning, which columns the model
is allowed to see (and which are held out as leakage), and the headline risk signal — the
**delinquency / default rate by year** across the whole loan book.

It reads a pre-computed statistics artifact so it renders instantly; regenerate that artifact with:

```bash
# representative 4% panel (fast; the deterministic hash sample is unbiased for rates)
python scripts/profile_fannie_dataset.py \
    --panel gs://sriram-credit-fm-data/output/raw/fannie_mae/panel_2000_2024.parquet \
    --out reports/fannie_dataset_profile.json

# OR the TRUE whole loan book, straight from the raw source
python scripts/profile_fannie_dataset.py \
    --raw-root gs://sriram-credit-fm-data/fannie_by_reporting \
    --out reports/fannie_dataset_profile.json --no-vintage
```

**Contents**
1. Setup &amp; overview
2. Column glossary (all 113 source fields + 6 derived)
3. Columns *included* in training vs *excluded* / *leakage*
4. Delinquency &amp; default rate by year
5. Per-column statistics
6. Notes &amp; caveats

## 1. Setup &amp; overview

In [1]:
import importlib.util
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 100)

# find the repo root (walk up until we see configs/)
ROOT = Path.cwd()
while not (ROOT / "configs" / "fannie_mae").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "configs" / "fannie_mae").exists(), "run inside the credit-foundation-model repo"

# import the glossary module directly (avoids importing credit_fm/__init__, which pulls in torch)
_gspec = importlib.util.spec_from_file_location(
    "fannie_glossary", ROOT / "src" / "credit_fm" / "data" / "fannie_glossary.py")
G = importlib.util.module_from_spec(_gspec)
_gspec.loader.exec_module(G)

# load the pre-computed statistics profile (may be absent on a fresh checkout)
PROFILE_PATH = ROOT / "reports" / "fannie_dataset_profile.json"
PROFILE = json.loads(PROFILE_PATH.read_text()) if PROFILE_PATH.exists() else None
print("glossary fields:", len(G.ALL_FIELDS), "| profile:",
      "loaded" if PROFILE else f"MISSING -> {PROFILE_PATH} (run profile_fannie_dataset.py)")

glossary fields: 119 | profile: MISSING -> /workspace/credit-foundation-model/reports/fannie_dataset_profile.json (run profile_fannie_dataset.py)


In [2]:
if PROFILE:
    print(f"source     : {PROFILE['source']}  ({PROFILE['source_kind']})")
    print(f"generated  : {PROFILE['generated_utc']}")
    print(f"rows       : {PROFILE['n_rows']:,}  (loan-months)")
    print(f"loans      : {PROFILE['n_loans']}")
    print(f"columns    : {PROFILE['n_columns']}")
    print(f"reporting  : {PROFILE['reporting_range'][0]} .. {PROFILE['reporting_range'][1]}")
    print(f"origination: {PROFILE['origination_range'][0]} .. {PROFILE['origination_range'][1]}")
else:
    print("No profile artifact yet — sections 4 and 5 will be empty until you generate it.")

No profile artifact yet — sections 4 and 5 will be empty until you generate it.


## 2. Column glossary

Every field in the dataset, condensed from the official *Single-Family Loan Performance Dataset and
Credit Risk Transfer — Glossary and File Layout* (© 2026 Fannie Mae). `position` is the 1-based
field position in the published layout; the last 6 rows (`position = derived`) are columns our
ingest step adds so the rest of the pipeline stays asset-generic.

In [3]:
def glossary_frame(fields):
    rows = []
    for name, (pos, plain, dtype, desc, enums) in fields.items():
        rows.append({"position": pos if pos is not None else "derived", "column": name,
                     "name": plain, "type": dtype, "description": desc,
                     "enumerations": enums or ""})
    df = pd.DataFrame(rows)
    # raw fields first (by position), derived last
    df["_sort"] = [p if isinstance(p, int) else 10_000 + i for i, p in enumerate(df["position"])]
    return df.sort_values("_sort").drop(columns="_sort").reset_index(drop=True)

GLOSSARY = glossary_frame(G.ALL_FIELDS)
print(f"{len(GLOSSARY)} fields ({len(G.RAW_FIELDS)} source + {len(G.DERIVED_FIELDS)} derived)")
GLOSSARY

119 fields (113 source + 6 derived)


,position,column,name,type,description,enumerations
0,1,reference_pool_id,Reference Pool ID,str,A unique identifier for the reference pool.,
1,2,loan_identifier,Loan Identifier,str,A unique identifier for the mortgage loan (does not correspond to loan identifiers in other Fann...,
2,3,monthly_reporting_period,Monthly Reporting Period,date(MMYYYY),The month/year of the servicer's cut-off period for the loan information (the observation month ...,
3,4,channel,Channel,str,The origination channel used by the party that delivered the loan to the issuer.,R = Retail; C = Correspondent; B = Broker
4,5,seller_name,Seller Name,str,The entity that delivered the mortgage loan to Fannie Mae. Sellers under ~1% of volume are shown...,
5,6,servicer_name,Servicer Name,str,The entity that serves as the primary servicer of the loan. Blank before Dec-2001; small service...,
6,7,master_servicer,Master Servicer,str,Fannie Mae.,
7,8,original_interest_rate,Original Interest Rate,float,The original interest rate on the loan as identified in the original mortgage note.,
8,9,current_interest_rate,Current Interest Rate,float,The rate of interest in effect for the periodic installment due (updated for modified loans).,
9,10,original_upb,Original UPB,float,The dollar amount of the loan as stated on the note at origination (rounded).,


**The 6 derived columns** (labels &amp; keys the pipeline computes) in detail:

In [4]:
glossary_frame(G.DERIVED_FIELDS)

,position,column,name,type,description,enumerations
0,derived,loan_id,Loan ID,str,"Renamed from loan_identifier (field 2). The entity key — splits are by loan_id, never by row.",
1,derived,reporting_date,Reporting Date (ISO),date(ISO),monthly_reporting_period (MMYYYY) parsed to an ISO 'YYYY-MM-DD' month-end string — the chronolog...,
2,derived,dlq_num,Delinquency Months (numeric),Int64,current_loan_delinquency_status cast to a number; 'XX'/blank -> <NA>. LEAKAGE (the outcome).,
3,derived,default_event,Default Event (label),boolean,"TRUE if dlq_num >= 6 (D180, 180+ days delinquent) OR zero_balance_code is a credit event (02/03/...",True = default; False = not; <NA> = unknown dlq
4,derived,prepay_event,Prepay Event,bool,"TRUE if zero_balance_code == '01' (prepaid or matured). A clean, non-credit termination.",True = prepaid/matured; False = not
5,derived,is_performing,Is Performing (gate),boolean,"TRUE if the loan is current (dlq_num == 0) and not yet terminated (no credit event, no prepay). ...",True = performing; False = not; <NA> = unknown dlq


## 3. Columns *included* in training vs *excluded* / *leakage*

The included/excluded split is **derived live** from `configs/fannie_mae/raw_schema.yaml` (all 113
source fields) and `configs/fannie_mae/baseline.yaml` (the `exclude` / `leakage` lists + the
id/time/label/gate roles), so this table can never drift from the config the model actually uses.

* **Model features** — everything not excluded and not leakage.
* **Excluded (non-features)** — ids, raw dates (superseded by derived ISO dates), high-cardinality
  geo, and non-tabular strings.
* **Leakage** — outcome / contemporaneous-state / post-default servicing columns. Using any of
  these would let the model peek at the answer, so they are dropped and the task is gated to
  loans *performing at the observation date* (predict **new** defaults).

In [5]:
def load_yaml(path):
    import yaml
    return yaml.safe_load(Path(path).read_text())

schema = load_yaml(ROOT / "configs" / "fannie_mae" / "raw_schema.yaml")
base = load_yaml(ROOT / "configs" / "fannie_mae" / "baseline.yaml")

all_cols = [c["name"] for c in schema["columns"]]
exclude = set(base.get("exclude", []))
leakage = set(base.get("leakage", []))
roles = {base.get("id_col"), base.get("time_col"), base.get("label_col"), base.get("gate_col")}
roles.discard(None)

features = [c for c in all_cols if c not in exclude and c not in leakage and c not in roles]
excluded = [c for c in all_cols if c in exclude]
leak_raw = [c for c in all_cols if c in leakage]

print(f"source fields          : {len(all_cols)}")
print(f"  model features       : {len(features)}")
print(f"  excluded (non-feature): {len(excluded)}")
print(f"  leakage (held out)   : {len(leak_raw)}")
print(f"  role cols (id/time/label/gate): {sorted(roles)}")

source fields          : 113
  model features       : 58
  excluded (non-feature): 15
  leakage (held out)   : 40
  role cols (id/time/label/gate): ['default_event', 'is_performing', 'loan_id', 'reporting_date']


### 3a. Model features (what the model *is* trained on)

In [6]:
pd.DataFrame([{"column": c, "name": G.ALL_FIELDS[c][1], "type": G.ALL_FIELDS[c][2],
               "description": G.ALL_FIELDS[c][3]} for c in features]).reset_index(drop=True)

,column,name,type,description
0,loan_identifier,Loan Identifier,str,A unique identifier for the mortgage loan (does not correspond to loan identifiers in other Fann...
1,channel,Channel,str,The origination channel used by the party that delivered the loan to the issuer.
2,original_interest_rate,Original Interest Rate,float,The original interest rate on the loan as identified in the original mortgage note.
3,current_interest_rate,Current Interest Rate,float,The rate of interest in effect for the periodic installment due (updated for modified loans).
4,original_upb,Original UPB,float,The dollar amount of the loan as stated on the note at origination (rounded).
5,upb_at_issuance,UPB at Issuance,float,The unpaid principal balance of the loan as of the cut-off date of the reference pool.
6,current_actual_upb,Current Actual UPB,float,"The current actual outstanding unpaid principal balance, reflecting payments received. Masked fo..."
7,original_loan_term,Original Loan Term,int,The number of months of regularly scheduled borrower payments at origination.
8,loan_age,Loan Age,int,The number of calendar months since origination (from the first full month interest accrues).
9,remaining_months_to_legal_maturity,Remaining Months to Legal Maturity,int,Calendar months remaining until the loan is due to be paid in full per the maturity date.


### 3b. Excluded non-feature columns (ids, raw dates, geo, non-tabular)

In [7]:
pd.DataFrame([{"column": c, "name": G.ALL_FIELDS[c][1], "why": G.ALL_FIELDS[c][3]}
              for c in excluded]).reset_index(drop=True)

,column,name,why
0,reference_pool_id,Reference Pool ID,A unique identifier for the reference pool.
1,monthly_reporting_period,Monthly Reporting Period,The month/year of the servicer's cut-off period for the loan information (the observation month ...
2,seller_name,Seller Name,The entity that delivered the mortgage loan to Fannie Mae. Sellers under ~1% of volume are shown...
3,servicer_name,Servicer Name,The entity that serves as the primary servicer of the loan. Blank before Dec-2001; small service...
4,master_servicer,Master Servicer,Fannie Mae.
5,origination_date,Origination Date,The date of the individual note (loan origination). Reparsed to an ISO date in this repo and use...
6,first_payment_date,First Payment Date,The date of the first scheduled loan payment by the borrower.
7,maturity_date,Maturity Date,The month/year the loan is scheduled to be paid in full per the loan documents.
8,metropolitan_statistical_area,Metropolitan Statistical Area (MSA),Numeric MSA/MSDA code for the property; '00000' if not in a designated area. High-cardinality ge...
9,zip_code_short,Zip Code Short,First three digits of the property ZIP code. High-cardinality geo — excluded from features.


### 3c. Leakage columns (outcome / contemporaneous / post-default — held out)

In [8]:
pd.DataFrame([{"column": c, "name": G.ALL_FIELDS[c][1], "why": G.ALL_FIELDS[c][3]}
              for c in leak_raw]).reset_index(drop=True)

,column,name,why
0,current_loan_delinquency_status,Current Loan Delinquency Status,Number of months the obligor is delinquent per the governing documents. Blank after removal; 'XX...
1,modification_flag,Modification Flag,Whether the loan has been modified (set to Y from the first modification onward). Post-default s...
2,zero_balance_code,Zero Balance Code,Reason the loan balance was reduced to zero or experienced a credit event. Drives the default/pr...
3,zero_balance_effective_date,Zero Balance Effective Date,Date the loan balance was reduced to zero. Leakage (termination timing).
4,upb_at_the_time_of_removal,UPB at the Time of Removal,The unpaid principal balance at the time of removal/liquidation. Leakage.
5,repurchase_date,Repurchase Date,Date a Reversed Credit Event Reference Obligation occurs. Leakage.
6,last_paid_installment_date,Last Paid Installment Date,Due date of the last paid installment collected. Post-default servicing — leakage.
7,foreclosure_date,Foreclosure Date,Date the legal action of foreclosure completed. Outcome — leakage.
8,disposition_date,Disposition Date,Date Fannie Mae's interest in the property ends. Outcome — leakage.
9,foreclosure_costs,Foreclosure Costs,"Expenses to obtain title, value, and maintain the property. Post-disposition — leakage."


## 4. Delinquency &amp; default rate by year

Two complementary views of credit risk across the whole book:

* **By reporting (calendar) year** — of all loan-months observed in year *Y*, the share that were
  30+ days past due, 180+ days past due (**D180**, our default threshold), in a **default_event**
  (D180 *or* a credit-event zero-balance code), and **performing**.
* **By vintage (origination year)** — of all loans *originated* in year *Y*, the share that **ever**
  hit a default_event over their observed life (loan-level lifetime default rate).

In [9]:
if PROFILE and PROFILE["delinquency_by_reporting_year"]:
    dlq = pd.DataFrame(PROFILE["delinquency_by_reporting_year"]).set_index("year")
    display(dlq)
else:
    dlq = None
    print("No profile — generate reports/fannie_dataset_profile.json first.")

No profile — generate reports/fannie_dataset_profile.json first.


In [10]:
if dlq is not None:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot(dlq.index, dlq["dpd30_plus_pct"], marker="o", label="30+ days past due")
    ax.plot(dlq.index, dlq["d180_plus_pct"], marker="s", label="180+ days (D180)")
    ax.plot(dlq.index, dlq["default_event_pct"], marker="^", label="default_event (D180 or credit ZBC)")
    ax.set_xlabel("reporting year")
    ax.set_ylabel("% of loan-months")
    ax.set_title("Delinquency & default rate by reporting year — whole loan book")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [11]:
if PROFILE and PROFILE["vintage_default_by_origination_year"]:
    vint = pd.DataFrame(PROFILE["vintage_default_by_origination_year"]).set_index("origination_year")
    display(vint)
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.bar(vint.index, vint["lifetime_default_pct"], color="#b3122f")
    ax.set_xlabel("origination (vintage) year")
    ax.set_ylabel("% of loans ever in default")
    ax.set_title("Lifetime default rate by vintage — whole loan book")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No vintage table (profile missing or built with --no-vintage).")

No vintage table (profile missing or built with --no-vintage).


### 4d. Is the 4% sample representative? — 4% panel vs 100% book

The pretraining panel is a **deterministic 4% hash sample on `loan_id`** (whole loan histories kept
or dropped together). This section proves that sampling doesn't distort the risk signal: it overlays
the sample's delinquency curve on the whole book's. Provide a second profile of the **full** book to
activate it:

```bash
python scripts/profile_fannie_dataset.py \
    --raw-root gs://sriram-credit-fm-data/fannie_by_reporting \
    --out reports/fannie_dataset_profile_full.json --delinquency-only --no-vintage --no-loan-count
```

The **pooled** (loan-month-weighted) default rate is the robust headline; per-year gaps in thin years
are just sampling noise.

In [12]:
FULL_PATH = ROOT / "reports" / "fannie_dataset_profile_full.json"
if PROFILE and FULL_PATH.exists():
    CMP = importlib.util.spec_from_file_location("cmp", ROOT / "scripts" / "compare_profiles.py")
    cmp = importlib.util.module_from_spec(CMP)
    CMP.loader.exec_module(cmp)
    full = json.loads(FULL_PATH.read_text())
    LA, LB = "4% sample", "100% book"
    yt = cmp._year_table(PROFILE, full, LA, LB)
    pa, pb = cmp._pooled(PROFILE), cmp._pooled(full)
    print(f"pooled default rate — {LA}: {pa['default_event_pct']}%   "
          f"{LB}: {pb['default_event_pct']}%   "
          f"(Δ {round(pa['default_event_pct'] - pb['default_event_pct'], 4)} pp, "
          f"{cmp._rel(pa['default_event_pct'], pb['default_event_pct'])}% rel)")
    display(yt[[f"default_event_pct__{LA}", f"default_event_pct__{LB}",
                "default_event_pct__diff_pp", "default_event_pct__diff_rel%"]])
else:
    yt = None
    print("Provide reports/fannie_dataset_profile_full.json (see the command above) to compare.")

Provide reports/fannie_dataset_profile_full.json (see the command above) to compare.


In [13]:
if yt is not None:
    import matplotlib.pyplot as plt
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2))
    yr = yt.index
    ax1.plot(yr, yt["default_event_pct__4% sample"], marker="o", label="4% sample")
    ax1.plot(yr, yt["default_event_pct__100% book"], marker="s", label="100% book")
    ax1.set_title("Default rate by year — sample vs book")
    ax1.set_xlabel("reporting year")
    ax1.set_ylabel("default_event %")
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    ax2.bar(yr, yt["default_event_pct__diff_pp"], color="#6a51a3")
    ax2.axhline(0, color="k", lw=0.8)
    ax2.set_title("Gap: sample − book (percentage points)")
    ax2.set_xlabel("reporting year")
    ax2.set_ylabel("Δ pp")
    ax2.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. Per-column statistics

Computed by `scripts/profile_fannie_dataset.py` in a single memory-bounded streaming pass over the
dataset. Numeric columns report min/mean/std and quantiles (quantiles from a 200k reservoir sample);
categorical columns report their top values; distinct counts are exact up to a 200k cap.

In [14]:
def stats_frames(profile):
    numeric, categ = [], []
    for name, s in profile["columns"].items():
        base = {"column": name, "n": s["n"], "nulls": s["nulls"], "null_%": s["null_pct"],
                "n_unique": s["n_unique"]}
        if s["kind"] == "numeric" and s.get("numeric"):
            numeric.append({**base, **s["numeric"]})
        else:
            top = s.get("top_values") or []
            base["min"], base["max"] = s.get("min"), s.get("max")
            base["top_values"] = ", ".join(f"{v}={p}%" for v, c, p in top[:6])
            categ.append(base)
    return (pd.DataFrame(numeric).set_index("column") if numeric else pd.DataFrame(),
            pd.DataFrame(categ).set_index("column") if categ else pd.DataFrame())

if PROFILE:
    NUM, CAT = stats_frames(PROFILE)
else:
    NUM = CAT = pd.DataFrame()
    print("No profile — nothing to show.")

No profile — nothing to show.


### 5a. Numeric columns

In [15]:
NUM if not NUM.empty else 'no numeric columns / no profile'

'no numeric columns / no profile'

### 5b. Categorical &amp; date columns (top values)

In [16]:
CAT if not CAT.empty else 'no categorical columns / no profile'

'no categorical columns / no profile'

### 5c. Missingness — most-null columns

In [17]:
if PROFILE and PROFILE["columns"]:
    miss = pd.DataFrame([{"column": n, "null_%": s["null_pct"]}
                         for n, s in PROFILE["columns"].items()])
    miss = miss.sort_values("null_%", ascending=False).set_index("column")
    display(miss.head(30))
    import matplotlib.pyplot as plt
    top = miss[miss["null_%"] > 0].head(25)
    if not top.empty:
        fig, ax = plt.subplots(figsize=(11, max(3, 0.32 * len(top))))
        ax.barh(top.index[::-1], top["null_%"][::-1], color="#4c72b0")
        ax.set_xlabel("% null")
        ax.set_title("Most-missing columns")
        plt.tight_layout()
        plt.show()

## 6. Notes &amp; caveats

* **Sampling & representativeness.** The default profile runs on the ingested **4% panel**, a
  *deterministic hash sample on `loan_id`* — whole loan histories are kept or dropped together, so
  observed rates are **unbiased estimates** of the whole book. Point `--raw-root` at the raw source
  for exact whole-book numbers.
* **Label definition.** `default_event = (dlq_num >= 6, i.e. D180) OR zero_balance_code ∈
  {02, 03, 09, 15}`. `is_performing` gates the task to loans current at the observation date so the
  model predicts **new** defaults, not ones already in progress.
* **Leakage discipline.** Section 3c columns are never features. Splits are **by `loan_id`** (never
  by row) and temporal by **origination date** — see `docs/data/fannie_mae.md` and the decision log.
* **Unknown delinquency.** `current_loan_delinquency_status = 'XX'` (or blank after removal) becomes
  `dlq_num = <NA>`; every downstream consumer resolves it with `.fillna(False)`.
* **Field availability drifts over time.** Some fields are populated only after certain releases
  (e.g. `*_classic_fico` from Dec-2025, several loss/servicing fields post-2020) — high null % on
  those columns is expected, not a data error.